# 02 — Model Integration with Ollama

Connect to local LLMs via the Ollama REST API. Build a reusable Python client for generation, chat, and embeddings.

**Concepts:** REST API, streaming aggregation, structured output parsing, HTTP client lifecycle

Requires Ollama running on `localhost:11434`.

---

## Definition

Ollama is a local inference server that exposes LLMs via a REST API. Instead of importing model weights into Python directly, you send HTTP requests to `http://localhost:11434` and get back JSON responses. This decouples the model runtime from your application code.

Three core endpoints:
- `POST /api/generate` — single-turn text generation (streaming or non-streaming)
- `POST /api/chat` — multi-turn conversation with message history
- `POST /api/embed` — text to vector embeddings

The `OllamaClient` class in `src/ollama_client.py` wraps these endpoints with Python methods, handling streaming aggregation, error handling, and resource cleanup.

## Motivation

Why not use Hugging Face Transformers or langchain?

| Approach | Pros | Cons |
|---|---|---|
| Hugging Face Transformers | Full model control, fine-tuning | Heavy RAM/VRAM, slow startup, complex dependency chain |
| LangChain | High-level abstractions, chains, agents | Heavy dependency tree, black-box internals, overkill for simple apps |
| **Direct Ollama API** | Lightweight (httpx only), full transparency, teach fundamentals | Manual streaming, no built-in chains |

Direct REST API is the best pedagogical choice — you learn request/response patterns, streaming, and error handling that generalize to any LLM API (OpenAI, Anthropic, etc.).

## Theory: REST API for LLMs

**Request flow:**

```
Python Client                     Ollama Server
     │                                │
     │── POST /api/generate ─────────→│
     │   {model, prompt, options}     │  Load weights
     │                                │  Run inference
     │←── SSE stream ────────────────│
     │   {"response":"The","done":false}  │
     │   {"response":" answer","done":false}│
     │   ...                          │
     │   {"response":"","done":true,    │
     │    "eval_count":42,               │
     │    "eval_duration":1234000000}    │
     │                                │
```

**Streaming:** Each line is a JSON object. The client concatenates `"response"` fields until `"done": true`. The final line contains metadata (token count, timing).

**Temperature:** Controls randomness (0.0 = deterministic, 1.0 = creative).

## Real-World Examples

| Application | Client Method | Model |
|---|---|---|
| Sentiment analysis | `generate()` | qwen3.5:2b |
| Summarization | `generate()` with system prompt | granite4.1:3b |
| Translation | `generate()` with structured prompt | translategemma:4b |
| Chatbot | `chat()` with message history | qwen3.5:4b |
| Document Q&A | `generate()` with document context | qwen3.5:4b |
| Semantic search | `embed()` to get vectors | qwen3-embedding:4b |

This project uses all of these across the 5 application tabs.

## Visual Explanation

**OllamaClient architecture:**

```
┌──────────────────────────────────────┐
│        OllamaClient                  │
│──────────────────────────────────────│
│  BASE = "http://localhost:11434"     │
│  _client: httpx.Client (reused)      │
├──────────────────────────────────────┤
│  +generate(model, prompt, **opts)    │
│    → POST /api/generate              │
│    → aggregates streaming lines      │
│    → returns {response, tokens, ...} │
│                                      │
│  +chat(model, messages, **opts)      │
│    → POST /api/chat                  │
│    → returns {message, usage}        │
│                                      │
│  +embed(model, text)                 │
│    → POST /api/embed                 │
│    → returns embedding vector        │
│                                      │
│  +measure_inference_time(…)          │
│    → calls generate, returns latency │
│                                      │
│  +close()                            │
│    → closes httpx client session     │
└──────────────────────────────────────┘
```

## Code Walkthrough

Each cell demonstrates one client method with real model calls (requires Ollama running).

In [ ]:
from src.ollama_client import OllamaClient

client = OllamaClient()
print(f"Client ready: {client.BASE}")

In [ ]:
# generate() — single-turn text generation
# Returns dict with "response" (text), "eval_count" (tokens), "eval_duration" (ns)
# temperature=0.0 makes output deterministic
result = client.generate("qwen3.5:2b", "Explain machine learning in one sentence.", temperature=0.0)
print(result["response"])

In [ ]:
# chat() — multi-turn conversation with message history
# Messages follow OpenAI format: [{"role": "user", "content": "..."}]
# Supports system, user, assistant roles
msgs = [
    {"role": "system", "content": "You are a poet."},
    {"role": "user", "content": "Write a haiku about Python."}
]
result = client.chat("qwen3.5:4b", msgs, temperature=0.7)
print(result["message"]["content"])

In [ ]:
# embed() — text to vector embeddings
# Returns a list of floats (embedding vector)
# qwen3-embedding:4b is a dedicated embedding model
vec = client.embed("qwen3-embedding:4b", "text embedding test")
print(f"Embedding dimension: {len(vec)}, first 5 values: {vec[:5]}")

In [ ]:
# measure_inference_time() — latency measurement
# Calls generate() and returns {"latency_s", "tokens", "tok_per_sec"}
m = client.measure_inference_time("qwen3.5:2b", "Hello world", temperature=0.0)
print(f"Latency: {m['latency_s']:.2f}s, Tokens: {m['tokens']}, Rate: {m['tok_per_sec']:.1f} tok/s")

In [ ]:
# Structured output with raw=True
# raw=True bypasses Ollama's system template — sends prompt directly
# Useful for JSON-structured prompting where you control the format
prompt = '{"sentiment": "positive"}'
result = client.generate("qwen3.5:2b", prompt, raw=True, temperature=0.0)
print(repr(result["response"]))

In [ ]:
# Always close the client to release HTTP connections
client.close()
print("Client closed.")

## Best Practices

1. **Create one client per module** — each tab creates and owns its own `OllamaClient` instance
2. **Always call `close()` or use context managers** — httpx connections can leak file descriptors
3. **Set `temperature=0.0` for deterministic tasks** — sentiment, summarization, translation benefit from consistency
4. **Handle connection errors** — Ollama may not be running; catch `httpx.ConnectError` gracefully
5. **Use `raw=True` for structured JSON output** — prevents Ollama from wrapping your prompt in a chat template
6. **Prefer `generate()` over `chat()` for single-turn tasks** — simpler, lower latency

## Common Mistakes

| Mistake | Fix |
|---|---|
| Not calling `client.close()` | Use `client.close()` or a context manager |
| Forgetting `temperature=0.0` for eval | Non-deterministic output makes tests flaky |
| Using `chat()` when `generate()` suffices | `chat()` requires message history management; `generate()` is simpler for single turns |
| Hardcoding `localhost:11434` | Make base URL configurable via env var or constructor param |
| Ignoring streaming chunks | Must accumulate response lines until `"done": true` — incomplete reads lose data |
| Passing system prompt via `chat()` when using `generate()` | `generate()` accepts `system=` parameter directly |

## Summary

- `/api/generate` for single-turn text generation (streaming lines → JSON)
- `/api/chat` for multi-turn conversations with message history
- `/api/embed` for vector embeddings
- `temperature` controls creativity (0.0 = deterministic)
- `raw` mode disables Ollama's template for direct prompt control
- Always close the HTTP client to free resources